In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# !pip install -q sentence-transformers

In [ ]:
# !pip install -q --force-reinstall "torch==2.5.1" "torchvision==0.20.1" "torchaudio==2.5.1"

In [ ]:
# import torch

# print("PyTorch:", torch.__version__)
# print("CUDA:", torch.version.cuda)
# print("GPU:", torch.cuda.get_device_name(0))
# print("Capability:", torch.cuda.get_device_capability(0))
# print("Architectures:", torch.cuda.get_arch_list())

In [ ]:
import pandas as pd
import numpy as np
import warnings
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# !pip uninstall -y torchcodec

In [ ]:
# movies = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv")
# print(movies.head())
# print(movies.info())
# print(movies.isnull().sum())

In [ ]:
# movies["text"] = (
#     movies["title"].fillna("") + " " +
#     movies["genres"].fillna("") + " " 
# )

In [ ]:
# movies["text"].head()

In [ ]:
def load_data():
    """
    specify dtypes explicitly to reduce memory usage.
    On a 25M row dataset, this saves ~500MB of RAM.
    """
    print("Loading MovieLens 25M dataset...")
    start = time.time()
    
    # Load ratings 
    ratings = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/ratings.csv",
        dtype={
            'userId': np.int32,
            'movieId': np.int32,
            'rating': np.float32,
            'timestamp': np.int64
        }
    )
    
    # Load movie metadata 
    movies = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv")
    
    # Load user-generated tags 
    tags = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/tags.csv",
        dtype={
            'userId': np.int32,
            'movieId': np.int32,
            'timestamp': np.int64
        }
    )
    
    # Convert timestamps to readable dates 
    ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
    ratings['year'] = ratings['datetime'].dt.year
    ratings['month'] = ratings['datetime'].dt.month
    ratings['day_of_week'] = ratings['datetime'].dt.dayofweek
    ratings['hour'] = ratings['datetime'].dt.hour
    
    elapsed = time.time() - start
    
    # Dataset statistics 
    n_users = ratings['userId'].nunique()
    n_items = ratings['movieId'].nunique()
    n_ratings = len(ratings)
    sparsity = 1 - n_ratings / (n_users * n_items)
    
    print(f"Loaded in {elapsed:.1f} seconds")
    print(f"")
    print(f"  Ratings:     {n_ratings:>12,}")
    print(f"  Users:       {n_users:>12,}")
    print(f"  Movies:      {n_items:>12,}")
    print(f"  Tags:        {len(tags):>12,}")
    print(f"  Sparsity:    {sparsity:>12.6f} ({sparsity*100:.2f}%)")
    print(f"")
    print(f"  Date range:  {ratings['datetime'].min().date()} to {ratings['datetime'].max().date()}")
    print(f"  Memory used: {ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB (ratings)")
    
    return ratings, movies, tags


ratings, movies, tags = load_data()

In [ ]:
# ============================================
# STEP 1: Basic preprocessing
# ============================================

print("Starting preprocessing...")

# Make a copy so the original ratings dataframe is preserved
ratings_clean = ratings.copy()

# Remove duplicate ratings
ratings_clean = ratings_clean.drop_duplicates(
    subset=['userId', 'movieId', 'timestamp']
)

# Remove missing values
ratings_clean = ratings_clean.dropna(
    subset=['userId', 'movieId', 'rating', 'timestamp']
)

# Check valid rating range
ratings_clean = ratings_clean[
    (ratings_clean['rating'] >= 0.5) &
    (ratings_clean['rating'] <= 5.0)
]

print(f"Original ratings:       {len(ratings):,}")
print(f"After cleaning:         {len(ratings_clean):,}")

print("\nRating distribution:")
print(ratings_clean['rating'].value_counts().sort_index())


In [ ]:
# ============================================
# STEP 1: Basic preprocessing
# ============================================

print("Starting preprocessing...")
print("-" * 50)

# Make a copy so the original ratings dataframe is preserved
ratings_clean = ratings.copy()

# Remove duplicate ratings, if any
duplicates = ratings_clean.duplicated(
    subset=['userId', 'movieId', 'timestamp']
).sum()

ratings_clean = ratings_clean.drop_duplicates(
    subset=['userId', 'movieId', 'timestamp']
)

# Remove missing values
missing_before = ratings_clean[['userId', 'movieId', 'rating', 'timestamp']].isnull().sum().sum()

ratings_clean = ratings_clean.dropna(
    subset=['userId', 'movieId', 'rating', 'timestamp']
)

# Check valid rating range
ratings_clean = ratings_clean[
    (ratings_clean['rating'] >= 0.5) &
    (ratings_clean['rating'] <= 5.0)
]

print(f"Original ratings:       {len(ratings):,}")
print(f"Duplicate rows removed: {duplicates:,}")
print(f"Missing values removed: {missing_before:,}")
print(f"After cleaning:         {len(ratings_clean):,}")

print("\nRating distribution:")
print(ratings_clean['rating'].value_counts().sort_index())


In [ ]:
# ============================================
# STEP 2: Filter users and movies
# ============================================

MIN_USER_RATINGS = 12
MIN_MOVIE_RATINGS = 64

# Count ratings per user
user_counts = ratings_clean['userId'].value_counts()

# Keep users with at least 5 ratings
valid_users = user_counts[
    user_counts >= MIN_USER_RATINGS
].index

# Count ratings per movie
movie_counts = ratings_clean['movieId'].value_counts()

# Keep movies with at least 5 ratings
valid_movies = movie_counts[
    movie_counts >= MIN_MOVIE_RATINGS
].index

# Filter dataset
ratings_filtered = ratings_clean[
    ratings_clean['userId'].isin(valid_users) &
    ratings_clean['movieId'].isin(valid_movies)
].copy()

print("Filtering results:")
print("-" * 50)
print(f"Ratings: {len(ratings_filtered):,}")
print(f"Users:   {ratings_filtered['userId'].nunique():,}")
print(f"Movies:  {ratings_filtered['movieId'].nunique():,}")


In [ ]:
# ============================================
# PREPARE MOVIE CONTENT
# ============================================

# --- Clean movies ---
movies_clean = movies.copy()

movies_clean["title"] = movies_clean["title"].fillna("").str.strip()
movies_clean["genres"] = movies_clean["genres"].fillna("")

# Convert genres from "Action|Comedy|Drama"
# to "Action Comedy Drama"
movies_clean["genres_text"] = (
    movies_clean["genres"]
    .str.replace("|", " ", regex=False)
)

# --- Process tags ---
tags_clean = tags.copy()

tags_clean = tags_clean.dropna(subset=["movieId", "tag"])

tags_clean["tag"] = (
    tags_clean["tag"]
    .astype(str)
    .str.strip()
)

tags_clean = tags_clean[tags_clean["tag"] != ""]

# Combine all tags belonging to each movie
movie_tags = (
    tags_clean
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.drop_duplicates()))
    .reset_index(name="tag")
)

# --- Merge tags into movies ---
movies_clean = movies_clean.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_clean["tag"] = movies_clean["tag"].fillna("")

# --- Create text for Transformer ---
movies_clean["year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)")
)

movies_clean["year"] = movies_clean["year"].fillna("unknown")

movies_clean["combined_text"] = (
    "Movie: " + movies_clean["title"] +
    ". Year: " + movies_clean["year"] +
    ". Genres: " + movies_clean["genres_text"] +
    ". Keywords: " + movies_clean["tag"]
)

# Check result
print(movies_clean[
    ["movieId", "title", "genres", "tag", "combined_text"]
].head(10))

In [ ]:
# ============================================
# STEP 3: Sort chronologically
# ============================================

ratings_filtered = ratings_filtered.sort_values(
    ['userId', 'timestamp']
).reset_index(drop=True)

print("Ratings sorted chronologically.")


In [ ]:
ratings_filtered = ratings_filtered.sort_values(
    ["userId", "timestamp"]
)

test_size = 0.2

train_df = (
    ratings_filtered
    .groupby("userId", group_keys=False)
    .apply(lambda x: x.iloc[:int(len(x) * (1 - test_size))])
)

test_df = (
    ratings_filtered
    .groupby("userId", group_keys=False)
    .apply(lambda x: x.iloc[int(len(x) * (1 - test_size)):])
)

In [ ]:
from itertools import combinations

# Only consider movies the user liked
positive_ratings = train_df[
    train_df["rating"] >= 4
].copy()

MAX_PAIRS_PER_USER = 20

pairs = []

for user_id, group in positive_ratings.groupby("userId"):

    movie_ids = group["movieId"].tolist()

    if len(movie_ids) < 2:
        continue

    user_pairs = list(
        combinations(movie_ids, 2)
    )

    user_pairs = user_pairs[:MAX_PAIRS_PER_USER]

    pairs.extend(user_pairs)

print("Positive pairs:", len(pairs))

In [ ]:
movie_text = dict(
    zip(
        movies_clean["movieId"],
        movies_clean["combined_text"]
    )
)

In [ ]:
training_pairs = []

for movie_a, movie_b in pairs:

    if movie_a not in movie_text:
        continue

    if movie_b not in movie_text:
        continue

    training_pairs.append({
        "anchor": movie_text[movie_a],
        "positive": movie_text[movie_b]
    })

print("Training pairs:", len(training_pairs))

In [ ]:
!pip install -q sentence-transformers datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(training_pairs)

train_dataset

In [ ]:
from sentence_transformers.losses import MultipleNegativesRankingLoss

loss = MultipleNegativesRankingLoss(model)

In [ ]:
# ============================================
# STEP 5: Create relevance labels
# ============================================

RELEVANCE_THRESHOLD = 4.0

train_df['relevant'] = (
    train_df['rating'] >= RELEVANCE_THRESHOLD
).astype(np.int8)

test_df['relevant'] = (
    test_df['rating'] >= RELEVANCE_THRESHOLD
).astype(np.int8)

print("Relevance distribution")
print("-" * 50)

print("Training:")
print(train_df['relevant'].value_counts())
print(
    f"Positive: {train_df['relevant'].mean()*100:.2f}%"
)

print("\nTesting:")
print(test_df['relevant'].value_counts())
print(
    f"Positive: {test_df['relevant'].mean()*100:.2f}%"
)

In [ ]:
# ============================================
# STEP 3: Generate movie embeddings
# ============================================

from sentence_transformers import SentenceTransformer

# Load pretrained Transformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Device:", model.device)

# Generate embeddings
embeddings = model.encode(
    movies_clean["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)

args = SentenceTransformerTrainingArguments(
    output_dir="movie-minilm-finetuned",

    num_train_epochs=2,

    per_device_train_batch_size=32,

    learning_rate=2e-5,

    warmup_steps=500,

    fp16=True,

    logging_steps=100,

    save_strategy="epoch"
)

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss
)

trainer.train()

In [ ]:
model.save_pretrained(
    "movie-minilm-finetuned"
)

In [ ]:
model = SentenceTransformer(
    "movie-minilm-finetuned"
)

In [ ]:
fine_tuned_embeddings = model.encode(
    movies_clean["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(fine_tuned_embeddings.shape)

In [ ]:
np.save(
    "movie_embeddings_finetuned.npy",
    fine_tuned_embeddings
)

In [ ]:
# ============================================
# STEP 4: Movie similarity search
# ============================================

from sklearn.metrics.pairwise import cosine_similarity

def get_similar_movies(movie_title, top_n=10):

    # Find movie
    matches = movies_clean[
        movies_clean["title"].str.contains(
            movie_title,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:
        print("Movie not found.")
        return

    # Use first matching movie
    movie_index = matches.index[0]

    # Get embedding
    query_embedding = embeddings[movie_index].reshape(1, -1)

    # Calculate similarity
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Get highest similarity scores
    similar_indices = similarities.argsort()[::-1]

    # Remove the movie itself
    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ]

    # Select top N
    similar_indices = similar_indices[:top_n]

    recommendations = movies_clean.iloc[similar_indices][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["similarity"] = similarities[similar_indices]

    return recommendations

In [ ]:
get_similar_movies("Toy Story", top_n=10)

In [ ]:
!pip install faiss-cpu

In [ ]:
# ============================================
# STEP 5: Build FAISS similarity index
# ============================================

import faiss
import numpy as np

# Make sure embeddings are float32

embeddings_f32 = fine_tuned_embeddings.astype("float32")

embedding_dim = embeddings_f32.shape[1]

index = faiss.IndexFlatIP(
    embedding_dim
)

index.add(embeddings_f32)

print(
    "Indexed movies:",
    index.ntotal
)

In [ ]:
def recommend_movies(movie_title, top_n=10):

    # Find movie
    matches = movies_clean[
        movies_clean["title"].str.contains(
            movie_title,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:
        print("Movie not found.")
        return

    movie_index = matches.index[0]

    # Get embedding
    query_vector = embeddings_f32[movie_index].reshape(1, -1)

    # Search FAISS index
    scores, indices = index.search(
        query_vector,
        top_n + 1
    )

    # Remove the query movie itself
    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == movie_index:
            continue

        results.append({
            "movieId": movies_clean.iloc[idx]["movieId"],
            "title": movies_clean.iloc[idx]["title"],
            "genres": movies_clean.iloc[idx]["genres"],
            "similarity": score
        })

        if len(results) == top_n:
            break

    return pd.DataFrame(results)

In [ ]:
recommend_movies("Toy Story", top_n=10)

In [ ]:
# Rating >= 4 means the user liked the movie
ratings_filtered["relevant"] = (
    ratings_filtered["rating"] >= 4
)

In [ ]:
# ============================================
# STEP 6: Temporal train/test split
# ============================================

ratings_sorted = ratings_filtered.sort_values(
    ["userId", "timestamp"]
)

# Last rating of each user → test set
test = (
    ratings_sorted
    .groupby("userId")
    .tail(1)
)

# Everything before the last rating → training set
train = ratings_sorted.drop(test.index)

print("Training ratings:", len(train))
print("Test ratings:", len(test))

In [ ]:
def hit_rate_at_k(user_id, k=10):

    # Movies the user liked in training
    user_train = train[
        (train["userId"] == user_id) &
        (train["rating"] >= 4)
    ]

    if len(user_train) == 0:
        return 0

    # Actual test movie
    user_test = test[
        (test["userId"] == user_id)
    ]

    if len(user_test) == 0:
        return 0

    test_movie = user_test.iloc[0]["movieId"]

    # Pick a movie the user liked previously
    seed_movie = user_train.iloc[-1]["movieId"]

    # Get its index
    matches = movies_clean.index[
        movies_clean["movieId"] == seed_movie
    ]

    if len(matches) == 0:
        return 0

    seed_index = matches[0]

    query_vector = embeddings_f32[
        seed_index
    ].reshape(1, -1)

    scores, indices = index.search(
        query_vector,
        k + 1
    )

    recommended_movie_ids = [
        movies_clean.iloc[idx]["movieId"]
        for idx in indices[0]
        if idx != seed_index
    ][:k]

    return int(test_movie in recommended_movie_ids)

In [ ]:
users = test["userId"].unique()

hits = []

for user_id in users[:1000]:
    hits.append(
        hit_rate_at_k(user_id, k=10)
    )

hit_rate = np.mean(hits)

print(f"Hit Rate@10: {hit_rate:.4f}")

In [ ]:
# ============================================
# STEP 7: MOVIE QUALITY / POPULARITY SCORE
# ============================================

movie_stats = (
    train
    .groupby("movieId")
    .agg(
        rating_mean=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

movie_stats.head()

In [ ]:
global_mean = train["rating"].mean()

print("Global mean rating:", global_mean)

In [ ]:
min_ratings = 50

In [ ]:
# Weighted rating

movie_stats["weighted_rating"] = (
    (
        movie_stats["rating_count"]
        /
        (movie_stats["rating_count"] + min_ratings)
    )
    * movie_stats["rating_mean"]
    
    +
    
    (
        min_ratings
        /
        (movie_stats["rating_count"] + min_ratings)
    )
    * global_mean
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

movie_stats["popularity_score"] = scaler.fit_transform(
    movie_stats[["weighted_rating"]]
)

In [ ]:
movie_stats[
    [
        "movieId",
        "rating_mean",
        "rating_count",
        "weighted_rating",
        "popularity_score"
    ]
].head(10)

In [ ]:
# ============================================
# STEP 8: HYBRID RECOMMENDER
# ============================================

CONTENT_WEIGHT = 0.8
POPULARITY_WEIGHT = 0.2


def hybrid_recommend(
    movie_title,
    top_n=10,
    content_weight=CONTENT_WEIGHT,
    popularity_weight=POPULARITY_WEIGHT
):
    
    # ----------------------------------------
    # 1. Find the input movie
    # ----------------------------------------
    
    matches = movies_clean[
        movies_clean["title"].str.contains(
            movie_title,
            case=False,
            na=False
        )
    ]
    
    if len(matches) == 0:
        print("Movie not found.")
        return pd.DataFrame()
    
    movie_index = matches.index[0]
    
    input_movie = movies_clean.loc[
        movie_index, "title"
    ]
    
    print("Input movie:", input_movie)
    
    
    # ----------------------------------------
    # 2. Get movie embedding
    # ----------------------------------------
    
    query_vector = embeddings_f32[
        movie_index
    ].reshape(1, -1)
    
    
    # ----------------------------------------
    # 3. FAISS similarity search
    # ----------------------------------------
    
    # Search for more than top_n because
    # some movies may not have rating statistics
    
    scores, indices = index.search(
        query_vector,
        min(top_n + 50, len(movies_clean))
    )
    
    
    # ----------------------------------------
    # 4. Create recommendation dataframe
    # ----------------------------------------
    
    recommendations = []
    
    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        
        # Skip the input movie
        if idx == movie_index:
            continue
        
        movie_id = movies_clean.iloc[
            idx
        ]["movieId"]
        
        recommendations.append({
            "movieId": movie_id,
            "content_score": float(score)
        })
    
    
    recommendations = pd.DataFrame(
        recommendations
    )
    
    
    # ----------------------------------------
    # 5. Add popularity information
    # ----------------------------------------
    
    recommendations = recommendations.merge(
        movie_stats[
            [
                "movieId",
                "rating_mean",
                "rating_count",
                "popularity_score"
            ]
        ],
        on="movieId",
        how="left"
    )
    
    
    # ----------------------------------------
    # 6. Handle movies without ratings
    # ----------------------------------------
    
    recommendations["popularity_score"] = (
        recommendations["popularity_score"]
        .fillna(0)
    )
    
    
    recommendations["rating_mean"] = (
        recommendations["rating_mean"]
        .fillna(global_mean)
    )
    
    recommendations["rating_count"] = (
        recommendations["rating_count"]
        .fillna(0)
    )
    
    
    # ----------------------------------------
    # 7. Normalize content score
    # ----------------------------------------
    
    recommendations["content_score_norm"] = (
        recommendations["content_score"]
        - recommendations["content_score"].min()
    ) / (
        recommendations["content_score"].max()
        - recommendations["content_score"].min()
        + 1e-8
    )
    
    
    # ----------------------------------------
    # 8. Calculate hybrid score
    # ----------------------------------------
    
    recommendations["hybrid_score"] = (
        content_weight *
        recommendations["content_score_norm"]
        +
        popularity_weight *
        recommendations["popularity_score"]
    )
    
    
    # ----------------------------------------
    # 9. Sort by hybrid score
    # ----------------------------------------
    
    recommendations = (
        recommendations
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
    )
    
    
    # ----------------------------------------
    # 10. Add movie information
    # ----------------------------------------
    
    recommendations = recommendations.merge(
        movies_clean[
            [
                "movieId",
                "title",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )
    
    
    # ----------------------------------------
    # 11. Final output
    # ----------------------------------------
    
    return recommendations[
        [
            "movieId",
            "title",
            "genres",
            "content_score",
            "rating_mean",
            "rating_count",
            "popularity_score",
            "hybrid_score"
        ]
    ]

In [ ]:
recommendations = hybrid_recommend(
    "Toy Story",
    top_n=10
)

recommendations

In [ ]:
weights = [
    (1.0, 0.0),
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.5, 0.5)
]

for content_weight, popularity_weight in weights:
    
    print(
        f"Content: {content_weight}, "
        f"Popularity: {popularity_weight}"
    )
    
    print(
        hybrid_recommend(
            "Toy Story",
            top_n=5,
            content_weight=content_weight,
            popularity_weight=popularity_weight
        )[["title", "hybrid_score"]]
    )
    
    print("-" * 60)

In [ ]:
def recommend_for_user(
    user_id,
    top_n=10,
    content_weight=0.8,
    popularity_weight=0.2
):
    
    # Movies the user liked
    liked_movies = train[
        (train["userId"] == user_id) &
        (train["rating"] >= 4)
    ]["movieId"].tolist()
    
    if len(liked_movies) == 0:
        return pd.DataFrame()
    
    all_recommendations = []
    
    # Use up to 5 liked movies as seeds
    seed_movies = liked_movies[-5:]
    
    for movie_id in seed_movies:
        
        matches = movies_clean.index[
            movies_clean["movieId"] == movie_id
        ]
        
        if len(matches) == 0:
            continue
        
        seed_index = matches[0]
        
        query_vector = embeddings_f32[
            seed_index
        ].reshape(1, -1)
        
        scores, indices = index.search(
            query_vector,
            51
        )
        
        for score, idx in zip(
            scores[0],
            indices[0]
        ):
            
            if idx == seed_index:
                continue
            
            recommended_movie_id = (
                movies_clean.iloc[idx]["movieId"]
            )
            
            all_recommendations.append({
                "movieId": recommended_movie_id,
                "content_score": float(score)
            })
    
    if not all_recommendations:
        return pd.DataFrame()
    
    recommendations = pd.DataFrame(
        all_recommendations
    )
    
    # If movie appears from multiple liked movies,
    # keep its strongest semantic score
    recommendations = (
        recommendations
        .groupby("movieId")
        ["content_score"]
        .max()
        .reset_index()
    )
    
    # Add popularity
    recommendations = recommendations.merge(
        movie_stats[
            [
                "movieId",
                "rating_mean",
                "rating_count",
                "popularity_score"
            ]
        ],
        on="movieId",
        how="left"
    )
    
    recommendations["popularity_score"] = (
        recommendations["popularity_score"]
        .fillna(0)
    )
    
    # Normalize content score
    recommendations["content_score_norm"] = (
        recommendations["content_score"]
        - recommendations["content_score"].min()
    ) / (
        recommendations["content_score"].max()
        - recommendations["content_score"].min()
        + 1e-8
    )
    
    # Hybrid score
    recommendations["hybrid_score"] = (
        content_weight *
        recommendations["content_score_norm"]
        +
        popularity_weight *
        recommendations["popularity_score"]
    )
    
    # Remove movies already watched
    watched = set(
        train[
            train["userId"] == user_id
        ]["movieId"]
    )
    
    recommendations = recommendations[
        ~recommendations["movieId"].isin(watched)
    ]
    
    recommendations = (
        recommendations
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
    )
    
    # Add movie information
    recommendations = recommendations.merge(
        movies_clean[
            [
                "movieId",
                "title",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )
    
    return recommendations

In [ ]:
user_id = test["userId"].iloc[0]

recommend_for_user(
    user_id,
    top_n=10
)

In [ ]:
def evaluate_hit_rate(
    user_ids,
    k=10,
    content_weight=0.8,
    popularity_weight=0.2
):
    
    hits = 0
    evaluated = 0
    
    for user_id in user_ids:
        
        if user_id not in test_relevant:
            continue
        
        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            popularity_weight=popularity_weight
        )
        
        if recommendations.empty:
            continue
        
        recommended_ids = set(
            recommendations["movieId"]
        )
        
        actual_ids = test_relevant[user_id]
        
        if recommended_ids & actual_ids:
            hits += 1
        
        evaluated += 1
    
    if evaluated == 0:
        return 0
    
    return hits / evaluated

In [ ]:
# Create the set of relevant movies for each user
# A rating >= 4 is considered a positive/relevant interaction

test_relevant = (
    test[test["rating"] >= 4]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Users with relevant test movies:", len(test_relevant))

In [ ]:
evaluation_users = list(
    test_relevant.keys()
)[:1000]

hit_rate = evaluate_hit_rate(
    evaluation_users,
    k=10
)

print(f"Hit Rate@10: {hit_rate:.4f}")

In [ ]:
for content_weight in [1.0, 0.9, 0.8, 0.7, 0.5]:
    
    popularity_weight = 1 - content_weight
    
    score = evaluate_hit_rate(
        evaluation_users,
        k=10,
        content_weight=content_weight,
        popularity_weight=popularity_weight
    )
    
    print(
        f"Content={content_weight:.1f}, "
        f"Popularity={popularity_weight:.1f}, "
        f"Hit Rate@10={score:.4f}"
    )